# Cyndx AI Engineering Challenge: MMR Search Algorithm

At Cyndx we develop AI and algorithms that perform search, information retrieval, and other related AI services with our large database of private companies. "Search quality" is a hard-to-define metric that we think about a lot. One potential way to improve the perceived quality of search results is to reduce redundancy and increase diversity in the results returned to the user. Maximal Marginal Relevance (MMR) is a simple algorithm that accomplishes this goal.

## The Challenge

Implement a basic search algorithm that utilizes MMR to re-rank the search results.

### Requirements
1. Implement the MMR algorithm defined in the original paper: https://www.cs.cmu.edu/~jgc/publication/The_Use_MMR_Diversity_Based_LTMIR_1998.pdf
2. Utilize the provided dataset `./companies.csv` to demonstrate your implementation
3. The search should be composed of two parts:

    * A basic search algorithm with the signature `def search(query: str, ...) -> pd.DataFrame` that returns relevancy-based results for the given query
    * An MMR re-ranking algorithm with the signature `def rerank(results: pd.DataFrame, ...) -> pd.DataFrame`, returning the optimized results
    * The demonstration should compose these functions, for example: `rerank(search(query, ...), ...)`
4. After you have implemented and demonstrated your algorithms, please answer the questions at the bottom of this notebook
5. Please provide a `README.md` file with instructions on how to run your code, and a `requirements.txt` file with the necessary dependencies
6. Email us your solution as a .zip file containing your modified notebook and all necessary files

In [1]:
!pip3 install -r requirements.txt

You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.10/bin/python3.10 -m pip install --upgrade pip' command.


In [2]:
# Basic Python libraries
import os
import pandas as pd
import numpy as np

# Used-defined libraries
from UDFs import clean_text
from UDFs import generate_companies_embeddings
from UDFs import mmr

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# sklearn libraries
from sklearn.metrics.pairwise import cosine_similarity

### Implementation

In [3]:
companies = pd.read_csv("./companies.csv")
companies.head()

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City
0,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh
1,Hunter Douglas NV,10001+,"Hunter Douglas NV engages in the design, manuf...",hunterdouglas.com,ZH,NL,Amsterdam/NL Metro,Rotterdam
2,"Root, Inc.",501-1000,"Root, Inc. is a technology insurance company, ...",inc.joinroot.com,OH,US,Columbus/OH Metro,Columbus
3,Bowlero Corp.,10001+,Bowlero Corp. engages in operating bowling cen...,bowlero.com,VA,US,Richmond/VA Metro,Mechanicsville
4,Meridia Real Estate III SOCIMI SA,1-10,Meridia Real Estate III SOCIMI SA operates as ...,meridiarealestateiiisocimi.com,CT,ES,Barcelona/Spain Metro,Barcelona


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
def search(query: str, df: pd.DataFrame, n=100) -> pd.DataFrame:
    # Clean the query
    query = clean_text(query)

    # Sentence Transformers can be used for more advanced search capabilities
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate embeddings for the companies
    df = generate_companies_embeddings(df, model)

    # Generate embeddings for the query
    query_embedding = model.encode([query])

    # Compute the cosine similarity between the query and the company embeddings
    df["similarity"] = df["embeddings"].apply(
        lambda x: cosine_similarity(query_embedding, [x])[0][0]
    )

    # Get the top n results based on similarity
    results = df.sort_values(by="similarity", ascending=False).reset_index(drop=True)
    results["query_embeddings"] = [query_embedding] * len(results)

    ### TODO: replace this with your own search algo:
    results = results.loc[:n, :].copy()

    return results

In [6]:
def rerank(results: pd.DataFrame, n=10) -> pd.DataFrame:

    # Query embeddings
    query_embeddings = results["query_embeddings"][0][0]

    # Extract document embeddings as a numpy array
    doc_embeddings = np.vstack(results["embeddings"].values)

    # Compute selected indices using MMR
    selected_indices = mmr(doc_embeddings, query_embeddings, n, 0.5)

    # Compute the MMR(Maximum Marginal Relevance) to rerank the results
    final_match = results.iloc[selected_indices, :].copy()

    return final_match

### Demo

In [7]:
rerank(search("companies who manufacture steel products", companies))

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,concatenated_name_description,concatenated_cleaned_text,embeddings,similarity,query_embeddings
0,United States Steel Corp.,10001+,United States Steel Corp. engages in the manuf...,ussteel.com,PA,US,Pittsburgh/PA Metro,Pittsburgh,United States Steel Corp. engages in the manuf...,united states steel corp engages in the manufa...,"[-0.031703609973192215, 0.01988265849649906, -...",0.687241,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
90,Associated Materials LLC,1001-5000,Associated Materials LLC engages in the manufa...,associatedmaterials.com,OH,US,Cleveland/OH Metro,Cuyahoga Falls,Associated Materials LLC engages in the manufa...,associated materials llc engages in the manufa...,"[-0.0164454635232687, -0.020316703245043755, -...",0.501850,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
93,Boryszew SA,5001-10000,Boryszew SA engages in the production and sale...,boryszew.com.pl,MZ,PL,Warsaw/Poland Metro,Warsaw,Boryszew SA engages in the production and sale...,boryszew sa engages in the production and sale...,"[-0.03041464276611805, -0.013747360557317734, ...",0.500397,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
5,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh,Molan Steel Co. engages in supplying steel pro...,molan steel co engages in supplying steel prod...,"[-0.10925497114658356, -0.013007495552301407, ...",0.631368,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
19,Insimbi Industrial Holdings Ltd.,501-1000,Insimbi Industrial Holdings Ltd. engages in th...,insimbi-group.co.za,GT,ZA,Johannesburg/S.Afr. Metro,Germiston,Insimbi Industrial Holdings Ltd. engages in th...,insimbi industrial holdings ltd engages in the...,"[-0.14633488655090332, 0.031998079270124435, -...",0.589827,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
74,"SIFCO Industries, Inc.",251-500,"SIFCO Industries, Inc. engages in the manufact...",sifco.com,OH,US,Cleveland/OH Metro,Cleveland,"SIFCO Industries, Inc. engages in the manufact...",sifco industries inc engages in the manufactur...,"[0.039052125066518784, -0.0672762393951416, -0...",0.515247,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
85,GrafTech International Ltd.,1001-5000,GrafTech International Ltd. engages in the man...,graftech.com,OH,US,Cleveland/OH Metro,Brooklyn Heights,GrafTech International Ltd. engages in the man...,graftech international ltd engages in the manu...,"[-0.13202665746212006, -0.01496044173836708, 0...",0.506430,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
20,Union Steel Holdings Ltd.,501-1000,Union Steel Holdings Ltd. is an investment hol...,unionsteel.com.sg,NW,SG,Singapore Metro,Singapore,Union Steel Holdings Ltd. is an investment hol...,union steel holdings ltd is an investment hold...,"[-0.09887104481458664, -0.021262140944600105, ...",0.589535,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
72,Termovent SC Livnica Celika AD,101-250,Termovent SC Livnica Celika AD engages in the ...,livnica.com,NB,RS,Belgrade/Serbia Metro,Backa Topola,Termovent SC Livnica Celika AD engages in the ...,termovent sc livnica celika ad engages in the ...,"[-0.003149702213704586, -0.03471856191754341, ...",0.516912,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
2,El Ezz Aldekhela Steel-Alexandria,1001-5000,El Ezz Aldekhela Steel-Alexandria engages in t...,NaN,GZ,EG,Africa (Northern) Metro,Giza,El Ezz Aldekhela Steel-Alexandria engages in t...,el ezz aldekhela steelalexandria engages in th...,"[-0.10446935147047043, 0.03445902094244957, -0...",0.650608,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."


### Questions

#### 1. Breifly summarize your implementation

- I designed a search system that helps users find relevant and diverse companies by combining semantic search with Maximal Marginal Relevance (MMR).

- First, I cleaned the company descriptions and used the SentenceTransformer model (all-MiniLM-L6-v2) to turn both the user’s search query and each company description into numerical vectors (called embeddings).

- I then used cosine similarity to rank companies based on how similar they are to the search query — this gave me a basic relevance ranking.

- After that, I applied the MMR algorithm to re-rank the top results. MMR balances relevance (how close the match is to the query) and diversity (how different the results are from each other), so users get a variety of useful results instead of a list of similar companies.

- All embeddings are stored inside the DataFrame, which keeps everything organized and allows fast computation.

- This two-step process ensures the user gets meaningful, non-repetitive search results that are still highly relevant to their query.

#### 2. What would be your next step(s) to increase performance?

- To improve the speed and efficiency of the system, I’d focus on the following:

- Precompute embeddings: Instead of calculating embeddings every time someone searches, I would generate and save all company embeddings in advance. This will make the system faster.

- Reduce embedding size: I could apply dimensionality reduction techniques like PCA or autoencoders to shrink the vector size, which would speed up similarity calculations.

- Use faster search libraries: Tools like FAISS or Annoy are great for quickly finding similar vectors, especially in large datasets.

- Cache frequent queries: If users often search similar terms (e.g., “AI companies” or “New York startups”), caching those results will avoid repeating calculations.

- Use parallel processing: When working with large numbers of companies, I’d use multi-threading or GPU support to handle multiple similarity checks at once.

All of these changes would make the system faster and more scalable as the dataset grows.

#### 3. What would be your next step(s) to improve search quality?

- To make the search results more useful and personalized for different types of users, here’s what I’d focus on:

- Incorporate user feedback: If users click on or save certain companies more often, we can learn from that behavior to improve future rankings.

- Handle location-based queries: If someone searches for “startups in Chicago,” I can use Named Entity Recognition (NER) to detect and match that location with company data, showing more relevant results.

- Add filters like employee count: Some users may want to explore only small startups or large companies. Adding options to filter by size (like employee count) would improve usability.

- Tune the MMR balance: Currently, MMR uses a fixed setting for relevance vs. diversity. I’d experiment with adjusting this based on the query type or even learning from users over time.

- Query expansion: I could expand the search terms with synonyms (e.g., “AI” and “artificial intelligence”) so users don’t miss relevant results just because of wording.

- Use more data fields: Right now, we’re using just the company description. In the future, I’d combine it with other data like industry, funding stage, or tags to improve accuracy.

These ideas would help make the search smarter and more tailored to what the user actually wants to find.